In [1]:
from transformers import AutoProcessor
import json

In [110]:
prompts = json.load(open("../../../curated_data/text/text_dataset/text_concept_red.json"))
vis = json.load(open("../../../activations/text/red_vis_direct_prompt.json"))
processor = AutoProcessor.from_pretrained("google/gemma-3-27b-it")

In [78]:
from circuitsvis.tokens import colored_tokens_multi
import torch



def visualize(features, prompts,n_batches, batch_size, layer,chosen_concept, start_from=0, system_prompt_token_count=0):
    pad = [0.0]*len(features[str(layer)][0][0][0])
    batched = features[str(layer)][start_from].copy()
    for i in range(start_from+1,start_from+n_batches):
        batched += features[str(layer)][i].copy()

    max_len = len(max(batched, key=lambda x: len(x)))
    for item in batched:
        if len(item)<max_len:
            for _ in range(max_len-len(item)):
                item.insert(0,pad.copy())        

    messages = [[
                    {
                        "role": "system",
                        "content": [{"type": "text", "text": f"You are a linguistics expert, your task is to identify all words that fall under the linguistic umbrella of {chosen_concept}, whether that manifests in direct words, nouns, pronouns, etc"}]
                    },
                    {
                        "role": "user",
                        "content": [
                            {"type": "text", "text": prompts[i]}
                        ]
                    }
                ] for i in range(start_from*batch_size, n_batches*batch_size+start_from*batch_size)]
    # print(list(range(start_from*batch_size, n_batches*batch_size+start_from*batch_size)), list(range(start_from+1,start_from+n_batches)))
    tokens = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=True,
                    return_dict=True, return_tensors="pt", padding=True)
    str_tokens = [processor.decode(t) for t in tokens["input_ids"][:,system_prompt_token_count:].flatten(end_dim=1)]
    # Visualize activations for top 20 most prominent features
    return colored_tokens_multi(str_tokens, torch.tensor(batched, dtype=torch.float32)[:,system_prompt_token_count:,:].flatten(end_dim=1))


In [79]:
#text_concept_a_dataset_with_diverse_colors,_with_some_bias_towards_green,_some_traditionally_green_things_like_trees_perhaps
#"a dataset with diverse colors, with some bias towards green, some traditionally green things like trees perhaps"

In [80]:
vis.keys()

dict_keys(['5', '10', '15', '20', '30', '35', '40', '50', '59'])

In [126]:
rendered = visualize(features=vis.copy(), prompts=prompts.copy(), n_batches=20, batch_size=2, layer=59,chosen_concept="a dataset with diverse colors, with some bias towards red, some traditionally red things like blood perhaps", start_from=170)

In [139]:
red_features_of_interest = {5: [49540, 16776, 57482, 79500, 37005, 34061, 4880, 48407, 57752, 61469, 16929, 41, 83755, 68524, 79023, 49329, 15283, 54206, 72382, 55102, 47560, 30286, 39637, 62806, 26848, 31714, 5859, 65124, 17133, 63981, 8178, 70268, 61181], 10: [35688, 55561, 56332, 77260, 81421, 45039, 33687, 946, 40051, 11158, 31191, 13277, 13687], 15: [69605, 18504, 58121, 26259, 26972], 20: [45536, 85084, 50525, 39], 30: [60423, 2185, 1932, 16782, 75534, 72853, 59290, 3877, 46250, 69038, 50994, 52917, 13626, 11720, 6216, 23504, 30038, 73560, 76760], 35: [13965, 5263, 61200, 74900, 27052, 79021, 62769, 82227, 54462, 45506, 35523, 7113, 28110, 8784, 21206, 60504, 58713, 41310, 5477, 79598, 82163, 51956, 46332], 40: [81160, 59154, 76436, 71449, 25760, 31027, 32055, 85180, 64966, 2504, 28746, 50385, 59864, 21337, 84063, 57185, 46947, 79078, 7533, 16623, 49402, 60667], 50: [30016, 36453, 76582, 41384, 49225, 4298, 10990, 73230, 32751, 37905, 76466, 23579, 61180], 59: [29024, 33669, 22855, 11209, 5035, 37036, 64333, 74542, 58283, 34411, 67217, 69107, 67412, 16405, 42200]}
green_features_of_interest = {5: [85376, 49540, 16776, 7561, 37005, 4880, 44944, 48407, 61469, 16929, 41, 83755, 68524, 79023, 55102, 72382, 14660, 47560, 5590, 26848, 64481, 31714, 5859, 38889, 61181, 16239, 77432, 70268, 20733], 10: [23490, 58754, 75012, 54732, 58892, 33687, 49041, 10963, 36919], 15: [53473, 36556, 69605, 26972], 20: [56282, 85084, 50525], 30: [37218, 3877, 60423, 6216, 1932, 16782, 75534, 23504, 50994, 73560, 52917, 76760], 35: [42240, 5263, 61200, 55829, 24219, 58659, 27052, 62769, 26557, 54462, 11582, 41024, 7113, 63818, 53835, 68303, 8784, 14419, 45016, 41310, 12648, 40042, 79598, 82163, 51956, 46332], 40: [73866, 76436, 71449, 83357, 21792, 29734, 71855, 32055, 64966, 2504, 28746, 60877, 50385, 52689, 25180, 84063, 57185, 85993, 56941, 16623, 26106, 60667], 50: [36453, 76582, 49225, 4298, 22734, 47726, 32751, 37905, 73230, 43603, 27870], 59: [29024, 33669, 52702, 7, 22855, 11209, 5035, 7212, 64333, 74542, 67217, 67412, 2904, 81854]}
blue_features_of_interest = {5: [49540, 49797, 7561, 37005, 4880, 44944, 48407, 45337, 61469, 41, 79023, 55102, 14660, 62806, 26848, 31714, 5859, 38889, 61181], 10: [35688, 68777, 31149, 26704, 13427, 10963, 28022, 33687], 15: [7041, 26259, 887, 66616, 13305], 20: [59377, 85084, 69782], 30: [3877, 60423, 11720, 6216, 1932, 16782, 23504, 43921, 50994, 52917, 16982, 76760, 73560, 13626, 17406], 35: [82314, 13965, 5263, 61200, 24219, 33828, 27052, 62769, 3260, 12349, 54462, 26557, 7113, 7115, 8784, 45016, 58713, 41310, 39655, 68202, 79598, 82163, 46835, 51956], 40: [38400, 43393, 75522, 18829, 76436, 74774, 29719, 28826, 21792, 25760, 30633, 64966, 2504, 28746, 50385, 25180, 29534, 84063, 57185, 85993, 56941, 5490, 21747, 26106, 60667], 50: [30016, 23299, 36453, 80742, 80838, 22734, 10990, 73230, 76466, 9751, 40506, 61180, 27870], 59: [29024, 20992, 22855, 11209, 74542, 67412, 16405]}

In [148]:
green = {5:[],10: [0, 2, 3, 4, 6, 8], 15:[],20: [1], 30: [9], 35: [2, 10, 11, 13, 15, 16, 20], 40: [0, 2, 4, 6, 9, 11, 13, 14, 15, 17, 18, 19, 20, 21], 50:[], 59:[]}
blue = {5:[1,13,16], 10:[3,4,6], 15:[0,3], 20:[0,1,2], 30:[7,10,12], 35:[3,5,8,13,14], 40:[0,12,20,22,23], 50:[9], 59:[]}
red = {5:[15,16,17,23,26], 10:[1,2,3,4,5,7,9,10,11,12], 15:[1,2,3], 20:[0,1,3], 30: [8, 17], 35: [2, 3, 5, 7, 9, 10, 12, 13, 14, 15, 16], 40:[12], 50: [6], 59:[]}

In [ ]:

green_feats = {key:[green_features_of_interest[key][index] for index in green[key]] for key in green.keys()}
red_feats = {key:[blue_features_of_interest[key][index] for index in blue[key]] for key in blue.keys()}
blue_feats = {key:[red_features_of_interest[key][index] for index in red[key]] for key in red.keys()}

In [154]:
with open("../../../feats/green.json", "w") as f:
    json.dump(green_feats, f)

In [155]:
with open("../../../feats/blue.json", "w") as f:
    json.dump(blue_feats, f)

In [156]:
with open("../../../feats/red.json", "w") as f:
    json.dump(red_feats, f)

In [ ]:
# These appear in the order that they appear in features of interest
#Green v2
#layer 20 feat 9
#layer 35 feat 9 
#layer 50 feat 17

#Blue v1
#layer 5 feat 1, 13, 16
#layer 10 feat 3, 4, 6
#layer 15 feat 0, 3
#layer 20 feat 0, 1, 2
#layer 30 feat 7, 10, 12
#layer 35 feat 3, 5, 8, 13, 14
#layer 40 feat 0, 12, 20, 22, 23
#layer 50 feat 9

#Red v1
#layer 5 feat 15, 16, 17, 23, 26
#layer 10 feat 1, 2, 3, 4, 5, 7, 9, 10, 11, 12
#layer 15 feat 1, 2, 3
#layer 20 feat (0, 1:colors), 3
#layer 40 feat 12



In [127]:
rendered

In [133]:
t = torch.rand((2,2,7))

In [134]:
t

tensor([[[4.8790e-01, 4.8663e-01, 6.7805e-01, 5.0194e-01, 1.0234e-01,
          5.1132e-01, 2.1480e-01],
         [2.7898e-02, 4.5834e-01, 9.9581e-01, 1.9950e-03, 3.3191e-01,
          1.5282e-01, 8.6629e-01]],

        [[1.7012e-01, 6.3327e-01, 9.0860e-01, 4.4126e-01, 9.6561e-01,
          1.8259e-02, 3.8499e-01],
         [3.7074e-05, 2.5775e-01, 6.9528e-01, 7.7547e-01, 5.1927e-01,
          4.1337e-01, 5.4419e-05]]])

In [135]:
t[:,:,[6,2]]

tensor([[[2.1480e-01, 6.7805e-01],
         [8.6629e-01, 9.9581e-01]],

        [[3.8499e-01, 9.0860e-01],
         [5.4419e-05, 6.9528e-01]]])

In [3]:
with open("../../../curated_data/text/text_dataset_classified/text_concept_green_classified.json", "r") as f:
    classified = json.load(f)

In [50]:
chosen_concept = "a dataset with diverse colors, with some bias towards green, some traditionally green things like the trees perhaps"
messages = [[
                {
                    "role": "system",
                    "content": [{"type": "text", "text": f"You are a linguistics expert, your task is to identify all words that fall under the linguistic umbrella of {chosen_concept}, whether that manifests in direct words, nouns, pronouns, etc"}]
                },
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompts[3]}
                    ]
                }
            ] ]
tokens = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=True,
                return_dict=True, return_tensors="pt", padding=True)
str_tokens = [processor.decode(t) for t in tokens["input_ids"].flatten(end_dim=1)]

In [46]:
tokens

{'input_ids': tensor([[     2,    105,   2364,    107,   3048,    659,    496, 127533,   7710,
         236764,    822,   4209,    563,    531,   8701,    784,   4171,    600,
           3798,   1208,    506,  45755,  36187,    529,    496,  15297,    607,
          12801,   7913, 236764,    607,   1070,  17482,   5645,   3826, 236764,
           1070,  37155,   3826,   2432,   1133,    506,   7146,   8229, 236764,
           3363,    600,  99688,    528,   1982,   4171, 236764,  77093, 236764,
         107701, 236764,   4044,    108,  32051,    586,  16715,  26677,  11595,
          53087,    506,    861,  51179,    618,    901,  28888,   4888, 236761,
          21516,   3305,    506,   2258,    691,    496,   3103,  16627, 236764,
            840,   4534,   9312,    506,  28239,   2778,    625,   6111,    531,
            506,   8351, 236761,   2195,   6544,    531,   6727,    625,    607,
           2173,  45204,    531,   7002,    506,  12732, 236761,    106,    107,
            10

In [51]:
token_acts = vis["20"][1][1]

In [49]:
classified['3']["labels"]

{'During': '0',
 ' autumn': '0',
 ',': '0',
 ' the': '0',
 ' once': '0',
 ' vibrant': '0',
 ' green': '1',
 ' canopy': '1',
 ' transforms': '0',
 ' into': '0',
 ' a': '0',
 ' sea': '0',
 ' of': '0',
 ' rus': '0',
 'set': '0',
 ' and': '0',
 ' gold': '0',
 '.': '0',
 ' Maple': '1',
 ' trees': '1',
 ' drop': '0',
 ' their': '0',
 ' fiery': '0',
 ' leaves': '1',
 ' blank': '0',
 'eting': '0',
 ' dark': '0',
 ' brown': '0',
 ' earth': '0',
 ' in': '0',
 ' crunchy': '0',
 ' carpet': '0',
 ' Squirrel': '0',
 's': '0',
 ' sc': '0',
 'urry': '0',
 ' about': '0',
 ' oblivious': '0',
 ' to': '0',
 ' dramatic': '0',
 ' shift': '0',
 ' landscape': '0',
 "'": '0',
 ' palette': '0'}

In [ ]:
for i in [i for i in range(len(token_acts)) if str_tokens[i] in classified['3']["labels"] and classified['3']["labels"][str_tokens[i]]=="1"]:
    print(token_acts[i])

[0.05167308449745178, 0.05422084778547287, 0.0, 0.05377448722720146, 0.0, 0.0, 0.053755734115839005, 0.05498645454645157, 0.0, 0.051969513297080994, 0.05248367041349411, 0.0, 0.05284348875284195, 0.05370872840285301, 0.05396881327033043, 0.05201283097267151, 0.052806347608566284, 0.05217823386192322, 0.055660929530858994, 0.05293485149741173]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0

In [8]:
import json
with open("../../../num_tokens_since_fired.json", "r") as f:
    bread = json.load(f)

In [9]:
len(bread["0"])

86016

In [10]:
nonzero = {i:sum(bread[i]) for i in bread.keys() if sum(bread[i])!=0}

In [11]:
len(nonzero)

2

In [12]:
max(nonzero.items(), key=lambda x: x[1])

('2', 8192)

In [7]:
nonzero

{'0': 16384,
 '1': 32768,
 '2': 8192,
 '4': 8192,
 '317': 8192,
 '352': 8192,
 '426': 8192,
 '436': 8192,
 '444': 8192,
 '453': 8192,
 '461': 8192,
 '462': 8192,
 '463': 8192,
 '470': 16384,
 '471': 16384,
 '473': 8192,
 '485': 8192,
 '487': 8192,
 '489': 8192,
 '490': 8192,
 '491': 16384,
 '492': 8192,
 '494': 8192,
 '496': 16384,
 '498': 8192,
 '500': 24576,
 '501': 49152,
 '502': 8192,
 '504': 8192,
 '506': 8192,
 '507': 8192,
 '510': 8192,
 '511': 16384,
 '519': 8192,
 '522': 8192,
 '523': 8192,
 '532': 16384,
 '533': 16384,
 '534': 8192,
 '535': 8192,
 '536': 8192,
 '537': 16384,
 '539': 8192,
 '543': 24576,
 '544': 16384,
 '547': 24576,
 '548': 16384,
 '552': 16384,
 '553': 16384,
 '556': 16384,
 '557': 8192,
 '558': 16384,
 '559': 16384,
 '560': 32768,
 '561': 65536,
 '562': 24576,
 '563': 16384,
 '564': 24576,
 '565': 8192,
 '568': 8192,
 '569': 8192,
 '570': 8192,
 '571': 16384,
 '572': 24576,
 '573': 16384,
 '574': 24576,
 '575': 8192,
 '576': 8192,
 '577': 8192,
 '578': 1638

In [96]:
sum(bread["2"])

254823

In [97]:
max(bread["2"])

3

In [98]:
max(bread["1"])

2

In [86]:
max(bread["0"])

8192

In [13]:
with open("../../../top_indices_BK_flattened.json", "r") as f:
    bread2 = json.load(f)

In [14]:
master_set = set()
for i in range(len(bread2)):
    master_set = master_set | set(bread2[i])

In [15]:
len(master_set)

86016

In [24]:
len(bread2[0])

8806400

In [25]:
8192*1075

8806400

In [31]:
len(set(bread2[4]))

86016

In [36]:
it = iter([0,1])

In [38]:
while(1):
    try:
        next(it)
    except StopIteration:
        print("done")
        break

done
